# 03. 피처 엔지니어링 분석

파생 피처 분포, 상관관계, VIF, Mutual Information 분석

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.figure_factory as ff
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import mutual_info_classif

from src.data.loader import load_gaming_behavior
from src.features.engineer import engineer_gaming_behavior_features, get_feature_columns
from src.features.selector import calculate_correlation_matrix, calculate_vif

In [2]:
df = load_gaming_behavior()
df = engineer_gaming_behavior_features(df)
print(f"Shape: {df.shape}")
df.head()

Shape: (40034, 22)


,PlayerID,Age,Gender,Location,GameGenre,PlayTimeHours,InGamePurchases,GameDifficulty,SessionsPerWeek,AvgSessionDurationMinutes,...,EngagementLevel,is_churned,playtime_per_session,weekly_activity_intensity,session_engagement_score,level_efficiency,achievement_rate,purchase_per_hour,age_group,activity_score
0,9000,43,Male,Other,Strategy,16.271119,0,Medium,6,108,...,Medium,0,2.711853,648,1.139334,4.855229,0.316456,0.000000,middle,27.481336
1,9001,29,Female,USA,Strategy,5.525961,0,Medium,5,144,...,Medium,0,1.105192,720,1.519112,1.990604,0.909091,0.000000,adult,7.357788
2,9002,22,Female,USA,Sports,8.223755,0,Easy,16,142,...,High,0,0.513985,2272,1.498013,4.255963,1.171429,0.000000,young_adult,22.467127
3,9003,35,Male,USA,Action,5.265351,1,Easy,9,85,...,Medium,0,0.585039,765,0.896698,10.825489,0.824561,0.189921,adult,25.079605
4,9004,33,Male,Europe,Action,15.531945,0,Medium,2,131,...,Medium,0,7.765972,262,1.381969,6.116427,0.389474,0.000000,adult,31.659583


## 1. 파생 피처 분포 (이탈 vs 유지)

In [3]:
derived_features = [
    "playtime_per_session", "weekly_activity_intensity",
    "session_engagement_score", "level_efficiency",
    "achievement_rate", "purchase_per_hour", "activity_score"
]

for feat in derived_features:
    fig = px.histogram(
        df, x=feat, color="is_churned", barmode="overlay",
        color_discrete_map={0: "#3498db", 1: "#e74c3c"},
        labels={"is_churned": "이탈 여부"},
        title=f"{feat} 분포 (이탈 vs 유지)",
        opacity=0.7
    )
    fig.show()

## 2. 상관관계 분석

In [4]:
feature_cols = get_feature_columns("kaggle")
numeric_features = feature_cols["numeric"]

corr_matrix = df[numeric_features].corr()

fig = ff.create_annotated_heatmap(
    z=corr_matrix.values.round(2),
    x=list(corr_matrix.columns),
    y=list(corr_matrix.index),
    colorscale="RdBu_r",
    showscale=True
)
fig.update_layout(title="피처 상관관계 히트맵", width=900, height=700)
fig.show()

In [5]:
to_drop = calculate_correlation_matrix(df[numeric_features], threshold=0.8)
print(f"상관관계 > 0.8인 제거 후보 피처: {to_drop}")

상관관계 > 0.8인 제거 후보 피처: ['session_engagement_score', 'purchase_per_hour', 'activity_score']


## 3. VIF (다중공선성) 분석

In [6]:
vif_df = calculate_vif(df[numeric_features])
print("VIF 분석 결과:")
print(vif_df.to_string())
print(f"\nVIF > 10 피처: {vif_df[vif_df['VIF'] > 10]['feature'].tolist()}")

VIF 분석 결과:
                      feature        VIF
1               PlayTimeHours        inf
2             SessionsPerWeek        inf
4                 PlayerLevel        inf
3   AvgSessionDurationMinutes        inf
5        AchievementsUnlocked        inf
9    session_engagement_score        inf
13             activity_score        inf
10           level_efficiency  10.517116
12          purchase_per_hour  10.516240
8   weekly_activity_intensity   7.487337
7        playtime_per_session   1.561943
11           achievement_rate   1.270074
6             InGamePurchases   1.000581
0                         Age   1.000160

VIF > 10 피처: ['PlayTimeHours', 'SessionsPerWeek', 'PlayerLevel', 'AvgSessionDurationMinutes', 'AchievementsUnlocked', 'session_engagement_score', 'activity_score', 'level_efficiency', 'purchase_per_hour']


## 4. Mutual Information 기반 피처 중요도

In [7]:
cat_features = feature_cols["categorical"]
df_enc = df.copy()
for col in cat_features:
    le = LabelEncoder()
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))

all_features = numeric_features + cat_features
X = df_enc[all_features]
y = df_enc["is_churned"]

mi_scores = mutual_info_classif(X, y, random_state=42)
mi_df = pd.DataFrame({"feature": all_features, "mi_score": mi_scores})
mi_df = mi_df.sort_values("mi_score", ascending=True)

fig = px.bar(mi_df, x="mi_score", y="feature", orientation="h",
             title="Mutual Information 피처 중요도",
             color="mi_score", color_continuous_scale="Viridis")
fig.update_layout(height=600)
fig.show()

## 5. 종합 비교

| 분석 방법 | 목적 | 결과 |
|-----------|------|------|
| 상관관계 | 다중공선성 피처 탐지 | 높은 상관 피처 쌍 식별 |
| VIF | 다중공선성 정량화 | VIF > 10 피처 확인 |
| Mutual Information | 타겟 변수와의 관련성 | 중요 피처 랭킹 |